# Fetch NMDC per-gene KO + EC annotations — manifest builder (issue #83)

This notebook **only builds the URL manifest**. It does no HTTP downloads
(those are too large to run in-kernel — the kernel kept dying mid-stream
during earlier attempts).

Three-stage pipeline:

1. **This notebook** (Spark kernel, fast) → queries `nmdc_metadata.data_object_set`
   and writes `loaded_ko_ec/manifest.csv`.
2. **`scripts/download_to_cache.py`** (terminal, hours) → reads `manifest.csv`,
   downloads each TSV with retries to `loaded_ko_ec/raw_cache/`. Resumable.
3. **`parse_ko_ec_cache.ipynb`** (any kernel, ~minutes) → reads the cache, parses
   BLAST-style TSVs, writes one Parquet per `data_object_type`. No network.

| `data_object_type` | ~files | ~total | ~avg | Format |
|---|---|---|---|---|
| `Annotation KEGG Orthology` | 4,776 | 163 GB | 35 MB | no-header BLAST TSV (11 cols) |
| `Annotation Enzyme Commission` | 4,771 | 111 GB | 24 MB | no-header BLAST TSV (11 cols) |

**Disk footprint:** ~274 GB of raw downloads will be cached under `OUT_DIR/raw_cache/`.
Override `DOWNLOAD_CACHE_DIR` to point at a scratch volume; delete the dir to
reclaim space after Parquet writes.


### Imports + config + logger + Spark

In [1]:
import logging
import os
from datetime import datetime
from pathlib import Path

OUT_DIR = Path(os.environ.get("KO_EC_OUT_DIR", "loaded_ko_ec"))
OUT_DIR.mkdir(exist_ok=True)

# Raw downloads will land here when scripts/download_to_cache.py runs.
# Override DOWNLOAD_CACHE_DIR to put them on a different volume.
CACHE_DIR = Path(os.environ.get("DOWNLOAD_CACHE_DIR", str(OUT_DIR / "raw_cache")))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR  = OUT_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)
LOG_FILE = LOG_DIR / f"fetch_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logger = logging.getLogger("nmdc_lakehouse.ko_ec.fetch")
logger.setLevel(logging.INFO)
for _h in list(logger.handlers):
    logger.removeHandler(_h)
_fh = logging.FileHandler(LOG_FILE)
_fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
logger.addHandler(_fh)
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(_sh)
logger.propagate = False

print(f"OUT_DIR:   {OUT_DIR.resolve()}")
print(f"CACHE_DIR: {CACHE_DIR.resolve()}")
print(f"LOG_FILE:  {LOG_FILE.resolve()}")

spark = get_spark_session(app_name="fetch_ko_ec_annotations")
print(f"Spark version: {spark.version}")


OUT_DIR:   /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_ko_ec
CACHE_DIR: /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_ko_ec/raw_cache
LOG_FILE:  /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_ko_ec/logs/fetch_20260429_174702.log
Spark version: 4.0.1


### Fetch the URL manifest from `data_object_set`

In [2]:
import time

import pandas as pd

from nmdc_lakehouse.file_types import resolve_file_types

TARGET_TYPES = resolve_file_types([
    "Annotation KEGG Orthology",
    "Annotation Enzyme Commission",
])

type_list = ", ".join(f"'{t}'" for t in TARGET_TYPES)

manifest_sql = f"""
SELECT id, url, data_object_type, was_generated_by, file_size_bytes, md5_checksum
FROM nmdc_metadata.data_object_set
WHERE data_object_type IN ({type_list})
  AND url IS NOT NULL
  AND url LIKE 'https://data.microbiomedata.org/%'
ORDER BY data_object_type, id
"""

t0 = time.monotonic()
manifest = spark.sql(manifest_sql).toPandas()
logger.info(f"manifest fetched in {time.monotonic() - t0:.1f}s")

# Dedup: data_object_set sometimes has multiple ids pointing at the same URL.
n_before = len(manifest)
manifest = manifest.drop_duplicates(subset=["url", "data_object_type"]).reset_index(drop=True)
if len(manifest) != n_before:
    logger.info(f"deduped: {n_before:,} → {len(manifest):,} (dropped {n_before - len(manifest):,} duplicates)")

logger.info(f"{len(manifest):,} unique data objects to process")
# file_size_bytes may be string-typed from the Parquet schema — cast defensively.
size_gb = pd.to_numeric(manifest["file_size_bytes"], errors="coerce").fillna(0).sum() / 1024**3
logger.info(f"size estimate: {size_gb:.1f} GB total")
print(manifest.groupby("data_object_type").size().to_string())

manifest fetched in 2.9s
9,696 unique data objects to process
size estimate: 274.3 GB total


data_object_type
Annotation Enzyme Commission    4848
Annotation KEGG Orthology       4848


### Save the manifest

Writes `manifest.csv` next to this notebook's output dir. The next cell prints the exact terminal command for `scripts/download_to_cache.py`.

In [3]:
manifest_csv = OUT_DIR / "manifest.csv"
manifest.to_csv(manifest_csv, index=False)
print(f"manifest written: {manifest_csv.resolve()}  ({len(manifest):,} rows)")

# Locate scripts/download_to_cache.py by walking up from cwd, so this works
# whether the notebook is opened from the repo root or from notebooks/.
def _find_repo_root() -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "scripts" / "download_to_cache.py").exists():
            return parent
    raise FileNotFoundError(
        "scripts/download_to_cache.py not found in any parent of "
        f"{Path.cwd()}"
    )

repo_root  = _find_repo_root()
downloader = repo_root / "scripts" / "download_to_cache.py"
log_path   = (OUT_DIR / "download.log").resolve()

print()
print("To run the standalone downloader, paste this into a terminal:")
print()
print(f"  nohup python {downloader} \\")
print(f"      --manifest {manifest_csv.resolve()} \\")
print(f"      --cache-dir {CACHE_DIR.resolve()} \\")
print(f"      --workers 8 \\")
print(f"      > {log_path} 2>&1 &")
print()
print(f"  tail -f {log_path}")


manifest written: /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_ko_ec/manifest.csv  (9,696 rows)

To run the standalone downloader, paste this into a terminal:

  nohup python /home/mamillerpa/nmdc-lakehouse/scripts/download_to_cache.py \
      --manifest /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_ko_ec/manifest.csv \
      --cache-dir /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_ko_ec/raw_cache \
      --workers 8 \
      > /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_ko_ec/download.log 2>&1 &

  tail -f /home/mamillerpa/nmdc-lakehouse/notebooks/loaded_ko_ec/download.log


## Next steps

The manifest is now on disk. Run, **from a terminal**:

```bash
cd /home/mamillerpa/nmdc-lakehouse
nohup python scripts/download_to_cache.py \
    --manifest notebooks/loaded_ko_ec/manifest.csv \
    --cache-dir notebooks/loaded_ko_ec/raw_cache \
    --workers 8 \
    > notebooks/loaded_ko_ec/download.log 2>&1 &

tail -f notebooks/loaded_ko_ec/download.log
```

When `download.log` reports it's done, open **`parse_ko_ec_cache.ipynb`** to
parse the cache and write the Parquets. Then run **`ingest_ko_ec_annotations.ipynb`**
to upload to MinIO bronze and register Delta tables under `nmdc_results`.
